# المرحلة الأولى: تجهيز بيانات اللهجات العربية

في هذا الدفتر نشرح كيف انتقلنا من الملفات الخام إلى بيانات موحدة قابلة للتدريب.

**المصادر المستخدمة في التدريب:** IADD وMADAR.  
**المصادر التي لم تدخل التدريب:** NADI المتاح TEST فقط، وQADI الذي يعتمد تدريبُه على Twitter IDs.


## أهداف هذه المرحلة

سنقوم بـ:

1. التعرف على بنية السجل الموحد.
2. تطبيق تطبيع النص العربي.
3. توحيد أسماء المناطق والدول.
4. إزالة السجلات المكررة.
5. فحص توزيع البيانات وتقسيماتها.

ملفات Python الأصلية التي تنفذ الدمج الكامل موجودة في `scripts/prepare_data.py`.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
PROCESSED = ROOT / "data" / "processed"
print("مجلد المشروع:", ROOT)
print("الملفات المعالجة:", sorted(path.name for path in PROCESSED.glob("*")))


## شكل السجل الموحد

كل نص أصبح مرتبطًا بالحقول التالية:

- `text`: النص المنظف.
- `region`: المنطقة.
- `country`: الدولة.
- `city`: المدينة إن كانت متوفرة.
- `source`: مصدر السجل.


In [ ]:
sample_path = PROCESSED / "all.jsonl"
with sample_path.open(encoding="utf-8") as handle:
    first_row = json.loads(next(handle))
first_row


## دالة تطبيع النص

التطبيع المستخدم في المشروع يزيل الروابط والتشكيل والتطويل، ويوحد بعض أشكال الحروف.
لا نستخدم Lemmatization أو Stopword Removal؛ لأن النموذج يعتمد لاحقًا على Character n-grams.


In [ ]:
import re

ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]")
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)

def normalize_text_demo(value):
    text = str(value or "")
    text = URL.sub(" ", text)
    text = text.replace("\u0640", "")
    text = ARABIC_DIACRITICS.sub("", text)
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    return re.sub(r"\s+", " ", text).strip()

examples = ["إزيــك؟ https://example.com", "مَرْحَبًا يا صديقي"]
[(text, normalize_text_demo(text)) for text in examples]


## قراءة البيانات وحساب الإحصاءات

التقرير المحفوظ يوضح حجم البيانات بعد كل مرحلة وتوزيعها حسب المصدر والمنطقة والدولة.


In [ ]:
report = json.loads((PROCESSED / "dataset_report.json").read_text(encoding="utf-8"))
summary = {
    "السجلات الخام": report["raw_rows"],
    "بعد المعالجة": report["after_preprocessing_rows"],
    "بعد إزالة التكرار": report["deduplicated_rows"],
    "التدريب": report["split_sizes"]["train"],
    "التحقق": report["split_sizes"]["validation"],
    "الاختبار": report["split_sizes"]["test"],
}
pd.Series(summary, name="القيمة")


In [ ]:
region_distribution = pd.Series(report["overall_distribution"]["region"]).sort_values(ascending=False)
region_distribution.to_frame("عدد النصوص")


## ملاحظة مهمة

المدن موجودة في بيانات MADAR، لكن هدف النسخة النهائية هو تصنيف المنطقة والدولة.
لذلك لم ندرب نموذج مدينة ولم نعرضها في الواجهة.
